# EGFR "Virtual Biopsy" — Multimodal Classifier (v2, upgraded)
### CT + clinical -> EGFR mutation (Mutant vs Wildtype)

**What changed from v1 (and why):**
- **Cache safety.** v1 silently reused old cached cubes, so `REQUIRE_SEG` had no effect. Now the cache folder is *stamped with the preprocessing settings*, and `FORCE_REBUILD` wipes it — changing settings genuinely rebuilds the data.
- **`REQUIRE_SEG=True` by default.** Only patients with a real tumour segmentation are used, so the image branch always sees the actual tumour (not a generic chest crop).
- **Pretrained 2.5D image branch.** Instead of a tiny 3D-CNN trained from scratch (which barely beat chance), we take 3 slices through the tumour and feed them to an **ImageNet-pretrained ResNet18**. Far better for small data. Switchable back to 3D via config.
- **Augmentation** (flips + intensity jitter) — train-time only, to fight overfitting.
- **Ethnicity added** to clinical features (a genuine EGFR risk factor, known at scan time — not cheating).
- **Cosine LR schedule + more epochs**, so longer training actually helps.
- **AIM semantic-feature baseline** kept as a *separate optional reference cell* (the ~0.89 ceiling), deliberately NOT wired into the main model — using it would just copy the radiologist's reading and make the image branch pointless.


## 1. Install dependencies (run once)

In [1]:
import sys
# !{sys.executable} -m pip install -q "pydicom<3" pydicom-seg SimpleITK scipy scikit-learn matplotlib tqdm torchvision
print("done")
# !pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

done


## 2. Imports

In [2]:
import os, glob, shutil, random, warnings, numpy as np, pandas as pd
from pathlib import Path
import pydicom, pydicom_seg, SimpleITK as sitk
from scipy import ndimage
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch", torch.__version__, "| Device:", DEVICE)

Torch 2.12.1+cu126 | Device: cuda


d:\git\PhDProjects\LungCancerGrading\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. CONFIG — all knobs here

In [3]:
# ---------- PATHS (EDIT) ----------
DATA_ROOT    = Path("data/nsclc_radiogenomics")
CLINICAL_CSV = Path("data/NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv")
AIM_DIR      = Path("data/AIM_files_updated-11-10-2020/AIM_files_updated-11-10-2020")   # optional, for the semantic baseline cell
CACHE_BASE   = Path("egfr_cache")
OUTPUT_DIR   = Path("egfr_out"); OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- TARGET ----------
TARGET_COL, POS_LABEL, NEG_LABEL = "EGFR mutation status", "Mutant", "Wildtype"

# ---------- IMAGE PREPROCESSING ----------
TARGET_SHAPE   = (32, 64, 64)     # (D,H,W) cached cube
CROP_PAD_VOX   = 16
HU_MIN, HU_MAX = -1000, 400
REQUIRE_SEG    = True             # only real-tumour crops
ALLOW_CENTER_CROP_FALLBACK = False
FORCE_REBUILD  = True             # wipe & rebuild the cache for the current settings (set False after first build)

# ---------- TABULAR ----------
INCLUDE_HISTOLOGY = False         # borderline-leaky; keep False
INCLUDE_ETHNICITY = True          # legitimate EGFR risk factor

# ---------- MODEL ----------
IMAGE_BACKBONE = "2.5d"           # "2.5d" (pretrained ResNet18) or "3d" (small CNN)
PRETRAINED     = True
IMG_FEAT_DIM   = 64
TAB_HIDDEN     = 64
FUSION_HIDDEN  = 64
DROPOUT        = 0.4

# ---------- TRAINING ----------
N_FOLDS = 5
EPOCHS  = 60
LR      = 1e-4                    # low, because the image backbone is pretrained
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 8
USE_AUGMENTATION  = True
USE_CLASS_WEIGHTS = True
SEED = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# cache folder stamped with preprocessing settings -> changing them rebuilds automatically
PRE_TAG = f"seg{int(REQUIRE_SEG)}_{TARGET_SHAPE[0]}x{TARGET_SHAPE[1]}x{TARGET_SHAPE[2]}_pad{CROP_PAD_VOX}"
CACHE_DIR = CACHE_BASE / PRE_TAG
if FORCE_REBUILD and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("cache:", CACHE_DIR)

cache: egfr_cache\seg1_32x64x64_pad16


## 4. Clinical labels + tabular features (now incl. ethnicity)

In [4]:
clin = pd.read_csv(CLINICAL_CSV)
clin["Case ID"] = clin["Case ID"].astype(str).str.strip()
clin["label"] = clin[TARGET_COL].map({POS_LABEL: 1, NEG_LABEL: 0})
clin = clin.dropna(subset=["label"]).copy(); clin["label"] = clin["label"].astype(int)
print("Usable EGFR labels:", len(clin)); print(clin["label"].value_counts().rename({0:NEG_LABEL,1:POS_LABEL}))

def build_tabular(df):
    f = pd.DataFrame(index=df.index)
    age = pd.to_numeric(df["Age at Histological Diagnosis"], errors="coerce")
    f["age"] = age.fillna(age.median())/100.0
    f["male"] = (df["Gender"].astype(str).str.strip().str.lower()=="male").astype(float)
    sm = df["Smoking status"].astype(str).str.strip()
    for v in ["Nonsmoker","Former","Current"]: f[f"smk_{v}"] = (sm==v).astype(float)
    for c in [c for c in df.columns if c.startswith("Tumor Location")]:
        key = c.split("choice=")[-1].rstrip(")").strip().replace(" ","_")
        f[f"loc_{key}"] = (df[c].astype(str).str.strip().str.lower()=="checked").astype(float)
    f = pd.concat([f, pd.get_dummies(df["%GG"].astype(str).str.strip(), prefix="gg").astype(float)], axis=1)
    if INCLUDE_ETHNICITY and "Ethnicity" in df.columns:
        f = pd.concat([f, pd.get_dummies(df["Ethnicity"].astype(str).str.strip(), prefix="eth").astype(float)], axis=1)
    if INCLUDE_HISTOLOGY:
        f = pd.concat([f, pd.get_dummies(df["Histology "].astype(str).str.strip(), prefix="hist").astype(float)], axis=1)
    return f

print("Tabular features:", build_tabular(clin).shape[1])

Usable EGFR labels: 172
label
Wildtype    129
Mutant       43
Name: count, dtype: int64
Tabular features: 25


## 5. Index DICOM images (CT + segmentation)

In [5]:
def scan_patient(pdir):
    info = {"ct_dir": None, "ct_n": 0, "seg_file": None, "seg_modality": None}
    for root, _, files in os.walk(pdir):
        dcms = [f for f in files if f.lower().endswith(".dcm")]
        if not dcms: continue
        try:
            ds = pydicom.dcmread(os.path.join(root, dcms[0]), stop_before_pixels=True, force=True)
        except Exception: continue
        mod = str(getattr(ds, "Modality", "")).upper()
        if mod == "CT":
            if len(dcms) > info["ct_n"]: info["ct_dir"], info["ct_n"] = root, len(dcms)
        elif mod in ("SEG","RTSTRUCT"):
            info["seg_file"], info["seg_modality"] = os.path.join(root, dcms[0]), mod
    return info

records = []
for cid in tqdm(clin["Case ID"].tolist(), desc="indexing"):
    pdir = DATA_ROOT/cid
    info = scan_patient(pdir) if pdir.exists() else {"ct_dir":None,"ct_n":0,"seg_file":None,"seg_modality":None}
    info["Case ID"] = cid; records.append(info)
idx = pd.DataFrame(records)
cohort = clin.merge(idx, on="Case ID", how="left")
print("CT:", cohort["ct_dir"].notna().sum(), "| SEG:", (cohort["seg_modality"]=="SEG").sum(),
      "| RTSTRUCT:", (cohort["seg_modality"]=="RTSTRUCT").sum(), "| neither:", cohort["seg_modality"].isna().sum())

indexing: 100%|██████████| 172/172 [00:24<00:00,  7.07it/s]

CT: 172 | SEG: 117 | RTSTRUCT: 0 | neither: 55


## 6. Preprocess -> tumour cubes (rebuilds because the cache is stamped + FORCE_REBUILD)

In [6]:
def load_ct(ct_dir):
    r = sitk.ImageSeriesReader(); r.SetFileNames(r.GetGDCMSeriesFileNames(str(ct_dir))); return r.Execute()

def seg_mask_on_ct(seg_file, ct_img):
    result = pydicom_seg.SegmentReader().read(pydicom.dcmread(seg_file))
    mask = None
    for num in result.available_segments:
        res = sitk.Resample(result.segment_image(num), ct_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
        a = sitk.GetArrayFromImage(res).astype(bool)
        mask = a if mask is None else (mask | a)
    return mask

def crop_to_cube(ct_arr, mask):
    if mask is not None and mask.any():
        zz,yy,xx = np.where(mask); p=CROP_PAD_VOX
        z0,z1=max(0,zz.min()-p),min(ct_arr.shape[0],zz.max()+p+1)
        y0,y1=max(0,yy.min()-p),min(ct_arr.shape[1],yy.max()+p+1)
        x0,x1=max(0,xx.min()-p),min(ct_arr.shape[2],xx.max()+p+1)
    else:
        Z,Y,X=ct_arr.shape; z0,z1=int(Z*.25),int(Z*.75);y0,y1=int(Y*.2),int(Y*.8);x0,x1=int(X*.2),int(X*.8)
    crop = ct_arr[z0:z1,y0:y1,x0:x1].astype(np.float32)
    crop = ndimage.zoom(crop, [t/max(1,s) for t,s in zip(TARGET_SHAPE,crop.shape)], order=1)
    crop = np.clip(crop, HU_MIN, HU_MAX)
    return ((crop-HU_MIN)/(HU_MAX-HU_MIN)).astype(np.float32)

status = []
for _, row in tqdm(cohort.iterrows(), total=len(cohort), desc="preprocess"):
    cid = row["Case ID"]; out = CACHE_DIR/f"{cid}.npy"
    if out.exists(): status.append((cid,"cached")); continue
    if pd.isna(row["ct_dir"]): status.append((cid,"no_ct")); continue
    try:
        ct = load_ct(row["ct_dir"]); ct_arr = sitk.GetArrayFromImage(ct)
        mask = None
        if row["seg_modality"]=="SEG":
            try: mask = seg_mask_on_ct(row["seg_file"], ct)
            except Exception: mask = None
        if mask is None or not mask.any():
            if REQUIRE_SEG or not ALLOW_CENTER_CROP_FALLBACK:
                status.append((cid,"no_seg_skipped")); continue
            np.save(out, crop_to_cube(ct_arr, None)); status.append((cid,"ok_centercrop")); continue
        np.save(out, crop_to_cube(ct_arr, mask)); status.append((cid,"ok_seg"))
    except Exception as e:
        status.append((cid, f"error:{type(e).__name__}"))

st = pd.DataFrame(status, columns=["Case ID","status"]); print(st["status"].value_counts())
cohort = cohort.merge(st, on="Case ID", how="left")
cohort = cohort[cohort["status"].isin(["ok_seg","ok_centercrop","cached"])].reset_index(drop=True)
print("\nFinal cohort:", len(cohort)); print(cohort["label"].value_counts().rename({0:NEG_LABEL,1:POS_LABEL}))

preprocess: 100%|██████████| 172/172 [14:38<00:00,  5.10s/it]

status
ok_seg            111
no_seg_skipped     61
Name: count, dtype: int64

Final cohort: 111
label
Wildtype    90
Mutant      21
Name: count, dtype: int64


## 7. Tabular matrix + Dataset (with train-time augmentation)

Augmentation (random flips + intensity jitter) is applied only when `train=True`, so the validation data is never altered.

In [7]:
tab_mat = build_tabular(cohort).values.astype(np.float32)
N_TAB = tab_mat.shape[1]; labels = cohort["label"].values.astype(np.float32)
print("Tabular dim:", N_TAB)

def augment_cube(cube):                       # cube: torch [1,D,H,W] in [0,1]
    if random.random()<0.5: cube = torch.flip(cube, dims=[2])   # flip H
    if random.random()<0.5: cube = torch.flip(cube, dims=[3])   # flip W
    cube = cube*random.uniform(0.9,1.1) + random.uniform(-0.05,0.05)   # intensity jitter
    return cube.clamp(0,1)

class EGFRDataset(Dataset):
    def __init__(self, indices, train=False):
        self.indices=list(indices); self.train=train
    def __len__(self): return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        cube = np.load(CACHE_DIR/f"{cohort.iloc[i]['Case ID']}.npy")
        img = torch.from_numpy(cube).float().unsqueeze(0)        # [1,D,H,W]
        if self.train and USE_AUGMENTATION: img = augment_cube(img)
        tab = torch.from_numpy(tab_mat[i]).float()
        y = torch.tensor([labels[i]], dtype=torch.float32)
        return img, tab, y

Tabular dim: 23


## 8. Model — pretrained 2.5D image branch + tabular MLP

**2.5D idea:** a CT tumour cube is 3D, but ImageNet models are 2D. We take 3 slices through the tumour (at 25%, 50%, 75% depth), treat them as the R/G/B channels of one image, resize to 224, and run a **pretrained ResNet18**. This reuses millions of learned 2D filters instead of learning 3D filters from ~110 tumours. Set `IMAGE_BACKBONE="3d"` to use the old small 3D-CNN instead.

In [8]:
class Image25DEncoder(nn.Module):
    def __init__(self, out_dim=IMG_FEAT_DIM, pretrained=PRETRAINED, p=DROPOUT):
        super().__init__()
        weights = tvm.ResNet18_Weights.DEFAULT if pretrained else None
        net = tvm.resnet18(weights=weights)
        self.features = nn.Sequential(*list(net.children())[:-1])    # -> [B,512,1,1]
        self.proj = nn.Sequential(nn.Flatten(), nn.Dropout(p), nn.Linear(512, out_dim), nn.ReLU(inplace=True))
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def forward(self, cube):                          # cube: [B,1,D,H,W]
        D = cube.shape[2]; idx = [D//4, D//2, (3*D)//4]
        x = cube[:,0][:, idx, :, :]                   # [B,3,H,W]
        x = F.interpolate(x, size=(224,224), mode="bilinear", align_corners=False)
        x = (x - self.mean)/self.std
        return self.proj(self.features(x))

class Image3DEncoder(nn.Module):
    def __init__(self, out_dim=IMG_FEAT_DIM, p=DROPOUT):
        super().__init__()
        def blk(ci,co): return nn.Sequential(nn.Conv3d(ci,co,3,padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True), nn.MaxPool3d(2))
        self.net = nn.Sequential(blk(1,16), blk(16,32), blk(32,64), nn.AdaptiveAvgPool3d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Dropout(p), nn.Linear(64,out_dim), nn.ReLU(inplace=True))
    def forward(self,x): return self.proj(self.net(x))

class TabularEncoder(nn.Module):
    def __init__(self, n_in, hidden=TAB_HIDDEN, out_dim=32, p=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in,hidden), nn.ReLU(inplace=True), nn.Dropout(p),
                                 nn.Linear(hidden,out_dim), nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)

class FusionNet(nn.Module):
    def __init__(self, n_tab, use_image=True, use_tabular=True):
        super().__init__(); self.use_image, self.use_tabular = use_image, use_tabular; dim=0
        if use_image:
            self.img = Image25DEncoder() if IMAGE_BACKBONE=="2.5d" else Image3DEncoder(); dim += IMG_FEAT_DIM
        if use_tabular:
            self.tab = TabularEncoder(n_tab); dim += 32
        self.head = nn.Sequential(nn.Linear(dim,FUSION_HIDDEN), nn.ReLU(inplace=True),
                                  nn.Dropout(DROPOUT), nn.Linear(FUSION_HIDDEN,1))
    def forward(self, img, tab):
        f=[]
        if self.use_image:   f.append(self.img(img))
        if self.use_tabular: f.append(self.tab(tab))
        return self.head(torch.cat(f,1))

print("FusionNet params:", sum(p.numel() for p in FusionNet(N_TAB).parameters()))

FusionNet params: 11219233


## 9. Train + predict (cosine LR schedule, AMP)

In [9]:
from torch.cuda.amp import autocast, GradScaler

@torch.no_grad()
def predict(model, loader):
    model.eval(); P,Y=[],[]
    for img,tab,y in loader:
        img,tab = img.to(DEVICE), tab.to(DEVICE)
        with autocast(enabled=(DEVICE=="cuda")): logit = model(img,tab)
        P.append(torch.sigmoid(logit).float().cpu().numpy().ravel()); Y.append(y.numpy().ravel())
    return np.concatenate(P), np.concatenate(Y)

def train_one(train_idx, val_idx, use_image=True, use_tabular=True):
    tr = DataLoader(EGFRDataset(train_idx, train=True),  batch_size=BATCH_SIZE, shuffle=True)
    va = DataLoader(EGFRDataset(val_idx,   train=False), batch_size=BATCH_SIZE, shuffle=False)
    model = FusionNet(N_TAB, use_image, use_tabular).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = GradScaler(enabled=(DEVICE=="cuda"))
    pw = None
    if USE_CLASS_WEIGHTS:
        n_pos = labels[train_idx].sum(); pw = torch.tensor([(len(train_idx)-n_pos)/max(1.0,n_pos)], device=DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    best_auc, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for img,tab,y in tr:
            img,tab,y = img.to(DEVICE), tab.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with autocast(enabled=(DEVICE=="cuda")): loss = crit(model(img,tab), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        p,yt = predict(model, va); auc = roc_auc_score(yt,p) if len(np.unique(yt))>1 else 0.5
        if auc>best_auc: best_auc=auc; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state); return model

## 10. Cross-validation + 11. Evaluation

In [10]:
def run_cv(use_image=True, use_tabular=True, tag="fusion"):
    strat = (cohort["label"].astype(str)+"_"+cohort["Patient affiliation"].astype(str)).values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(cohort)); fa=[]
    for f,(tr,va) in enumerate(skf.split(np.arange(len(cohort)), strat)):
        m = train_one(tr, va, use_image, use_tabular)
        p,yt = predict(m, DataLoader(EGFRDataset(va), batch_size=BATCH_SIZE)); oof[va]=p
        a = roc_auc_score(yt,p) if len(np.unique(yt))>1 else 0.5; fa.append(a)
        print(f"[{tag}] fold {f} AUROC {a:.3f}")
    print(f"[{tag}] mean fold AUROC {np.mean(fa):.3f} +/- {np.std(fa):.3f}"); return oof

def evaluate(oof, name):
    y = labels.astype(int)
    auroc, auprc = roc_auc_score(y,oof), average_precision_score(y,oof)
    fpr,tpr,thr = roc_curve(y,oof); t = thr[np.argmax(tpr-fpr)]
    tn,fp,fn,tp = confusion_matrix(y,(oof>=t).astype(int)).ravel()
    print(f"=== {name} ===\n  AUROC {auroc:.3f} | AUPRC {auprc:.3f} (prevalence {y.mean():.2f})")
    print(f"  @thr {t:.2f}: sensitivity {tp/(tp+fn):.2f} specificity {tn/(tn+fp):.2f}")
    print(f"  confusion [[TN {tn}, FP {fp}],[FN {fn}, TP {tp}]]")
    return dict(auroc=auroc, auprc=auprc)

oof_fusion = run_cv(True, True, "fusion")
_ = evaluate(oof_fusion, "FUSION (CT + clinical)")

[fusion] fold 0 AUROC 0.778
[fusion] fold 1 AUROC 0.750
[fusion] fold 2 AUROC 0.583
[fusion] fold 3 AUROC 0.431
[fusion] fold 4 AUROC 0.750
[fusion] mean fold AUROC 0.658 +/- 0.133
=== FUSION (CT + clinical) ===
  AUROC 0.626 | AUPRC 0.306 (prevalence 0.19)
  @thr 0.49: sensitivity 0.57 specificity 0.74
  confusion [[TN 67, FP 23],[FN 9, TP 12]]


## 12. Ablation — fusion vs image-only vs clinical-only

The decisive comparison. With the pretrained image branch + clean tumour crops, watch whether **image-only** rises above v1's ~0.61 and whether **fusion** now beats clinical-only.

In [11]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
oof_lr = cross_val_predict(lr, tab_mat, labels.astype(int), cv=N_FOLDS, method="predict_proba")[:,1]
oof_img  = run_cv(True,  False, "image-only")
oof_clin = run_cv(False, True,  "clinical-only")
print()
for nm,o in [("clinical-only (logreg)",oof_lr),("clinical-only (deep)",oof_clin),
             ("image-only (deep)",oof_img),("FUSION",oof_fusion)]:
    print(f"{nm:24s} AUROC {roc_auc_score(labels.astype(int),o):.3f}  AUPRC {average_precision_score(labels.astype(int),o):.3f}")

[image-only] fold 0 AUROC 0.733
[image-only] fold 1 AUROC 0.611
[image-only] fold 2 AUROC 0.569
[image-only] fold 3 AUROC 0.514
[image-only] fold 4 AUROC 0.806
[image-only] mean fold AUROC 0.647 +/- 0.107
[clinical-only] fold 0 AUROC 0.822
[clinical-only] fold 1 AUROC 0.778
[clinical-only] fold 2 AUROC 0.667
[clinical-only] fold 3 AUROC 0.521
[clinical-only] fold 4 AUROC 0.806
[clinical-only] mean fold AUROC 0.719 +/- 0.113

clinical-only (logreg)   AUROC 0.651  AUPRC 0.308
clinical-only (deep)     AUROC 0.589  AUPRC 0.252
image-only (deep)        AUROC 0.590  AUPRC 0.335
FUSION                   AUROC 0.626  AUPRC 0.306


## 13. Save + single-patient inference

In [12]:
all_idx = np.arange(len(cohort))
final_model = train_one(all_idx, all_idx[:max(2,len(all_idx)//5)], True, True)
torch.save(final_model.state_dict(), OUTPUT_DIR/"egfr_fusion_v2.pt"); print("saved")

@torch.no_grad()
def predict_patient(case_id):
    i = cohort.index[cohort["Case ID"]==case_id][0]
    img = torch.from_numpy(np.load(CACHE_DIR/f"{case_id}.npy")).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    tab = torch.from_numpy(tab_mat[i]).float().unsqueeze(0).to(DEVICE)
    final_model.eval()
    return {"Case ID":case_id, "P(EGFR-mutant)":round(torch.sigmoid(final_model(img,tab)).item(),3),
            "true_label": POS_LABEL if labels[i]==1 else NEG_LABEL}
print(predict_patient(cohort.iloc[0]["Case ID"]))

saved
{'Case ID': 'R01-003', 'P(EGFR-mutant)': 0.607, 'true_label': 'Mutant'}


## 14. (OPTIONAL, reference only) AIM semantic-feature ceiling

The AIM XML files hold the radiologist's hand-scored tumour descriptors. A model on these reached ~0.89 AUROC in the original paper. **This is a reference ceiling, NOT part of our model** — it predicts EGFR from a human's reading of the CT, which would make our image branch pointless. Run it only to see "how far expert features can go." Best-effort parser; may need tweaks to your AIM schema.

In [21]:
# %% [markdown]
# ## 14. (OPTIONAL, reference only) AIM semantic-feature ceiling
# 
# Parses the radiologist's hand-scored tumour descriptors from the AIM XML files.
# Uses regex to safely bypass messy XML schema namespaces.

# %%
import os
import glob
import re
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

def parse_aim_characteristics(xml_text):
    feats = {}
    # Find all characteristic blocks, ignoring line breaks
    blocks = re.findall(r'<[iI]magingObservationCharacteristic.*?>(.*?)</[iI]magingObservationCharacteristic>', xml_text, re.DOTALL)
    for b in blocks:
        # Extract the semantic label
        lbl_match = re.search(r'<label\s+value="([^"]+)"', b)
        # Extract all typeCode codeSystems
        cs_matches = re.findall(r'<typeCode[^>]*codeSystem="([^"]+)"', b)
        
        if lbl_match and cs_matches:
            # Filter out generic 'RadLex' schema tags, keep the actual human-readable value
            val = [c for c in cs_matches if c.strip() and "radlex" not in c.lower()]
            if val:
                feats[lbl_match.group(1).strip()] = val[0].strip()
    return feats

if AIM_DIR.exists():
    rows = {}
    all_xml_files = glob.glob(os.path.join(str(AIM_DIR), "*.xml"))
    
    for cid in cohort["Case ID"]:
        matching_files = [f for f in all_xml_files if cid in os.path.basename(f)]
        if matching_files:
            # Read as raw string, bypassing xml.etree completely
            with open(matching_files[0], 'r', encoding='utf-8', errors='ignore') as f:
                res = parse_aim_characteristics(f.read())
            if res:
                rows[cid] = res

    print(f"Successfully parsed matching AIM profiles for {len(rows)} out of {len(cohort)} patients.")

    if rows:
        aim_df = pd.DataFrame.from_dict(rows, orient="index")
        if not aim_df.empty and aim_df.shape[1] > 0:
            aim_oh = pd.get_dummies(aim_df.astype(str)).reindex(cohort["Case ID"]).fillna(0).values.astype(float)
            
            if aim_oh.shape[1] > 0:
                lr2 = make_pipeline(StandardScaler(with_mean=False), LogisticRegression(max_iter=2000, class_weight="balanced"))
                oof_aim = cross_val_predict(lr2, aim_oh, labels.astype(int), cv=N_FOLDS, method="predict_proba")[:,1]
                print("\n=== BASELINE RESULTS ===")
                print(f"AIM semantic features -> AUROC: {roc_auc_score(labels.astype(int), oof_aim):.3f} | Total feature columns: {aim_oh.shape[1]}")
            else:
                print("Zero dummy variables generated.")
        else:
            print("No valid features extracted.")
    else:
        print("No feature dictionary keys compiled.")
else:
    print(f"AIM_DIR not found at {AIM_DIR}.")

Successfully parsed matching AIM profiles for 0 out of 111 patients.
No feature dictionary keys compiled.


In [19]:
# %% [markdown]
# ## Path and Filename Diagnostic

# %%
import os
import glob

print("=== PATH VERIFICATION ===")
print(f"Configured AIM_DIR: {AIM_DIR}")
print(f"Absolute path: {AIM_DIR.resolve()}")
print(f"Does directory exist? {AIM_DIR.exists()}")

if AIM_DIR.exists():
    all_xmls = glob.glob(os.path.join(str(AIM_DIR), "*.xml"))
    print(f"\nTotal XML files found in directory: {len(all_xmls)}")
    
    if len(all_xmls) > 0:
        print("\n=== FILENAME VS CASE ID PREVIEW ===")
        print("First 5 XML filenames on disk:")
        for f in all_xmls[:5]:
            print(f"  -> {os.path.basename(f)}")
            
        print("\nFirst 5 Case IDs in your cohort DataFrame:")
        for cid in cohort["Case ID"].head(5):
            print(f"  -> '{cid}'")
            
        print("\n=== MATCHING TEST ===")
        sample_cid = cohort["Case ID"].iloc[0]
        print(f"Testing match for Case ID: '{sample_cid}'")
        
        # Test 1: Exact substring match
        exact_matches = [f for f in all_xmls if sample_cid in os.path.basename(f)]
        print(f"Exact substring match found: {len(exact_matches)} files")
        
        # Test 2: Case-insensitive match
        lower_matches = [f for f in all_xmls if sample_cid.lower() in os.path.basename(f).lower()]
        print(f"Case-insensitive match found: {len(lower_matches)} files")
        
        # Test 3: Removing hyphens (e.g., 'AMC-001' vs 'AMC001')
        nohyphen_matches = [f for f in all_xmls if sample_cid.replace("-", "") in os.path.basename(f).replace("-", "")]
        print(f"Hyphen-ignored match found: {len(nohyphen_matches)} files")
        
else:
    print("\nSTOP: The directory does not exist. Check your DATA_ROOT or AIM_DIR string.")

=== PATH VERIFICATION ===
Configured AIM_DIR: data\AIM_files_updated-11-10-2020\AIM_files_updated-11-10-2020
Absolute path: D:\git\PhDProjects\LungCancerGrading\data\AIM_files_updated-11-10-2020\AIM_files_updated-11-10-2020
Does directory exist? True

Total XML files found in directory: 190

=== FILENAME VS CASE ID PREVIEW ===
First 5 XML filenames on disk:
  -> AMC-003.xml
  -> AMC-005.xml
  -> AMC-006.xml
  -> AMC-007.xml
  -> AMC-008.xml

First 5 Case IDs in your cohort DataFrame:
  -> 'R01-003'
  -> 'R01-004'
  -> 'R01-005'
  -> 'R01-006'
  -> 'R01-007'

=== MATCHING TEST ===
Testing match for Case ID: 'R01-003'
Exact substring match found: 1 files
Case-insensitive match found: 1 files
Hyphen-ignored match found: 1 files


## 15. Reading the results / what to try if image is still weak

- **Did image-only improve** vs v1 (~0.61)? If yes, pretrained 2.5D + clean crops worked. If it's still flat, the EGFR signal in *this* small cohort's images may simply be weak — an honest finding.
- **Does fusion beat clinical-only?** That's the headline. Even a small consistent gain (with overlapping-but-higher AUPRC) supports multimodality.
- **More to try:** unfreeze/lower-LR fine-tuning, more slices (mean-pool many slices instead of 3), 3D pretrained encoder (MedicalNet), stronger augmentation, or adding the AIM features knowingly (accepting it becomes a "predict-from-radiologist-reading" model).
- **Don't chase 0.9.** On ~170 patients with 25% positives, robust AUROC ~0.75-0.82 with fusion adding a real margin over clinical-only is a strong, honest result.
